# DocPrompting Pipeline (NLP Team 19 2026 Version)
Team 19 - UIT
**Repo:** https://github.com/buuhq-uit/team19-docprompting



## 1. Clone Repo Github & Cài đặt Môi trường
Tải repo mà nhóm đã chuẩn hóa (sửa lỗi Dependency, Monkey Patching) và cài đặt các thư viện mới nhất.

In [33]:
%cd /content
!rm -rf team19-docprompting

# Clone repo đã được fork và fix lỗi thư viện
!git clone https://github.com/buuhq-uit/team19-docprompting.git

%cd /content/team19-docprompting
!pip install -r requirements.txt


/content
Cloning into 'team19-docprompting'...
remote: Enumerating objects: 251, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 251 (delta 17), reused 24 (delta 10), pack-reused 215 (from 1)
Receiving objects: 100% (251/251), 48.84 MiB | 24.37 MiB/s, done.
Resolving deltas: 100% (91/91), done.
/content/team19-docprompting


## 2. Liên kết Dữ liệu (Symlink)
Liên kết 2 thư mục `data` và `models` từ Google Drive sang thư mục code để chạy siêu nhanh mà không cần tốn dung lượng copy.

In [ ]:
# connect google drive để thấy thư mục docprompting_assets download sẵn data của tác giả
# 2 file: docprompting_data.zip và docprompting_generator_models.zip
# File Data (129MB): 👉 https://drive.google.com/file/d/1CzNlo8-e4XqrgAME5zHEWEKIQMPga0xl/view
# File Model (5.7GB): 👉 https://drive.google.com/file/d/1NmPMxY1EOWkjM7S8VSKa13DKJmEZ3TqV/view
from google.colab import drive
drive.mount('/content/drive')

In [34]:
# Xóa thư mục rỗng mặc định của Github
!rm -rf /content/team19-docprompting/data
!rm -rf /content/team19-docprompting/models
# Tạo link với Google Drive
!ln -s /content/drive/MyDrive/docprompting_assets/data /content/team19-docprompting/data
!ln -s /content/drive/MyDrive/docprompting_assets/models /content/team19-docprompting/models
!ls -la /content/team19-docprompting/models/generator/codet5-base

total 871599
-rw------- 1 root root      1531 Dec  2  2022 config.json
-rw------- 1 root root    294364 Dec  2  2022 merges.txt
-rw------- 1 root root 891641279 Dec  2  2022 pytorch_model.bin
-rw------- 1 root root      3083 Dec  2  2022 special_tokens_map.json
-rw------- 1 root root        25 Dec  2  2022 tokenizer_config.json
-rw------- 1 root root    575045 Dec  2  2022 vocab.json


## 3. Dense Retriever

In [35]:
# Tìm kiếm Top 10 tài liệu mô tả API Python liên quan nhất cho từng câu hỏi
!PYTHONPATH=/content/team19-docprompting python retriever/simcse/run_inference.py \
  --model_name "neulab/docprompting-codet5-python-doc-retriever" \
  --source_file data/conala/conala_nl.txt \
  --target_file data/conala/python_manual_firstpara.tok.txt \
  --source_embed_save_file data/conala/.tmp/src_embedding \
  --target_embed_save_file data/conala/.tmp/tgt_embedding \
  --sim_func cls_distance.cosine \
  --num_layers 12 \
  --save_file data/conala/retrieval_results.json

{
  "model_name": "neulab/docprompting-codet5-python-doc-retriever",
  "batch_size": 48,
  "source_file": "data/conala/conala_nl.txt",
  "target_file": "data/conala/python_manual_firstpara.tok.txt",
  "source_embed_save_file": "data/conala/.tmp/src_embedding",
  "target_embed_save_file": "data/conala/.tmp/tgt_embedding",
  "save_file": "data/conala/retrieval_results.json",
  "top_k": 200,
  "cpu": false,
  "pooler": "cls",
  "log_level": "verbose",
  "nl_cm_folder": "data/conala/nl.cm",
  "sim_func": "cls_distance.cosine",
  "num_layers": 12,
  "origin_mode": false,
  "oracle_eval_file": "data/conala/cmd_dev.oracle_man.full.json",
  "eval_hit": false,
  "normalize_embed": false,
  "source_idx_file": "data/conala/conala_nl.id",
  "target_idx_file": "data/conala/python_manual_firstpara.tok.id"
}
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--neulab--docprompting-codet5-python-doc-retriever/snapshots/5d546757de07391fe953dbdfe69f690a59a4e65e/confi

## 4. Chạy FiD Code Generator
Không cần bọc (Monkey patch) mã nguồn như phiên bản cũ vì repo Github đã được sửa tận gốc. Batch size được thiết lập ở mức an toàn (2) cho GPU T4.

In [36]:
!PYTHONPATH=/content/team19-docprompting python generator/fid/test_reader_simple.py \
    --model_path models/generator/conala.fid.codet5.top10/checkpoint/best_dev/best_dev \
    --tokenizer_name models/generator/codet5-base \
    --eval_data data/conala/fid.cmd_test.codet5.t10.json \
    --per_gpu_batch_size 2 \
    --n_context 10 \
    --name conala.fid.codet5.top10 \
    --checkpoint_dir models/generator \
    --result_tag test_same \
    --main_port 81692

0 - Number of nodes: 1
0 - Node ID        : 0
0 - Local rank     : 0
0 - Global rank    : 0
0 - World size     : 1
0 - GPUs per node  : 1
0 - Multi-node     : False
0 - Multi-GPU      : False
0 - Hostname       : a5e3c156d8ef
[04/19/2026 08:38:23] {test_reader_simple.py:133} INFO - load the tokenizer from codet5
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Loading weights: 100% 260/260 [00:00<00:00, 938.90it/s, Materializing param=shared.weight]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are p

## 5. Xem Kết Quả Sinh Code

In [37]:
import json
import os

eval_data_path = '/content/team19-docprompting/data/conala/fid.cmd_test.codet5.t10.json'
id_to_question = {}
if os.path.exists(eval_data_path):
    with open(eval_data_path, 'r', encoding='utf-8') as f:
        eval_data = json.load(f)
        id_to_question = {item['id']: item['question'] for item in eval_data}

result_path = "/content/team19-docprompting/models/generator/conala.fid.codet5.top10/test_results_test_same.json"
if os.path.exists(result_path):
    data = []
    with open(result_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))

    print("Hiển thị 3 kết quả Code đầu tiên:\n")
    for item in data[:3]:
        q_id = item.get('question_id')
        print("--- Ý định NL (Intent):", id_to_question.get(q_id, "Không rõ"))
        print("--- Code Đáp án (Gold):", item.get('gold'))
        print("+++ Code AI (Generated):", item.get('clean_code'))
        print("-" * 60)
else:
    print("File kết quả chưa được tạo ra.")

Hiển thị 3 kết quả Code đầu tiên:

--- Ý định NL (Intent): divide the values with same keys of two dictionary `d1` and `d2`
--- Code Đáp án (Gold): {k: (float(d2[k]) / d1[k]) for k in d2}
+++ Code AI (Generated): {k: d1[k] / d2[k] for k, v in list(d1.items())}
------------------------------------------------------------
--- Ý định NL (Intent): divide values associated with each key in dictionary `d1` from values associated with the same key in dictionary `d2`
--- Code Đáp án (Gold): dict((k, float(d2[k]) / d1[k]) for k in d2)
+++ Code AI (Generated): {d1[k] / d2[k] for k, v in list(d1.items())}
------------------------------------------------------------
--- Ý định NL (Intent): download "http://randomsite.com/file.gz" from http and save as "file.gz"
--- Code Đáp án (Gold): testfile = urllib.request.URLopener() testfile.retrieve('http://randomsite.com/file.gz', 'file.gz')
+++ Code AI (Generated): urllib.request.urlretrieve('http://randomsite.com/file.gz', 'http://randomsite.com/file.gz'